1. Setup

In [2]:
from src.bronze.pipeline import BronzePipeline

pipeline = BronzePipeline()
spark = pipeline.spark

26/06/24 19:36:04 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


2. CSV

In [2]:
csv_output_path = pipeline.run(
    source_path="/app/data/raw_local/RAW/2024-07-13_0800/CONTROLE DE MEDICOES E PAGAMENTOS/ControleMedicoesPagamentos.csv",
    source_type="csv",
    dataset_name="controle_medicoes_pagamentos",
    snapshot_date="2024-07-13_0800",
    source_file="ControleMedicoesPagamentos.csv",
    file_hash="manual_test_hash"
)

print(csv_output_path)
spark.read.parquet(csv_output_path).select("_snapshot_date", "_source_file", "_source_type").show(5)

26/06/24 19:09:31 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
26/06/24 19:09:32 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


s3a://contracts/bronze/source_type=csv/dataset=controle_medicoes_pagamentos/snapshot_date=2024-07-13_0800/
+---------------+--------------------+------------+
| _snapshot_date|        _source_file|_source_type|
+---------------+--------------------+------------+
|2024-07-13_0800|ControleMedicoesP...|         csv|
|2024-07-13_0800|ControleMedicoesP...|         csv|
|2024-07-13_0800|ControleMedicoesP...|         csv|
|2024-07-13_0800|ControleMedicoesP...|         csv|
|2024-07-13_0800|ControleMedicoesP...|         csv|
+---------------+--------------------+------------+
only showing top 5 rows



3. XLSX

In [3]:
xlsx_output_path = pipeline.run(
    source_path="/app/data/raw_local/RAW/2024-07-13_0800/CONTROLE DE MEDICOES EM ANDAMENTO/Exportação_bm_acompanhamento.xlsx",
    source_type="xlsx",
    dataset_name="controle_medicoes_andamento",
    snapshot_date="2024-07-13_0800",
    source_file="Exportação_bm_acompanhamento.xlsx",
    file_hash="manual_test_hash"
)

print(xlsx_output_path)
spark.read.parquet(xlsx_output_path).select("_snapshot_date", "_source_file", "_source_type").show(5)

s3a://contracts/bronze/source_type=xlsx/dataset=controle_medicoes_andamento/snapshot_date=2024-07-13_0800/
+---------------+--------------------+------------+
| _snapshot_date|        _source_file|_source_type|
+---------------+--------------------+------------+
|2024-07-13_0800|Exportação_bm_aco...|        xlsx|
|2024-07-13_0800|Exportação_bm_aco...|        xlsx|
|2024-07-13_0800|Exportação_bm_aco...|        xlsx|
|2024-07-13_0800|Exportação_bm_aco...|        xlsx|
|2024-07-13_0800|Exportação_bm_aco...|        xlsx|
+---------------+--------------------+------------+
only showing top 5 rows



4. XLSB / NACT

In [4]:
xlsb_output_path = pipeline.run(
    source_path="/app/data/raw_local/RAW/2024-07-13_0800/NACT/202211_ADMIN.xlsb",
    source_type="xlsb",
    dataset_name="nact",
    snapshot_date="2024-07-13_0800",
    source_file="202211_ADMIN.xlsb",
    file_hash="manual_test_hash"
)

print(xlsb_output_path)
spark.read.parquet(xlsb_output_path).select("_snapshot_date", "_source_file", "_source_type").show(5)

26/06/24 19:09:54 WARN TaskSetManager: Stage 6 contains a task of very large size (1043 KiB). The maximum recommended task size is 1000 KiB.


s3a://contracts/bronze/source_type=xlsb/dataset=nact/snapshot_date=2024-07-13_0800/
+---------------+-----------------+------------+
| _snapshot_date|     _source_file|_source_type|
+---------------+-----------------+------------+
|2024-07-13_0800|202211_ADMIN.xlsb|        xlsb|
|2024-07-13_0800|202211_ADMIN.xlsb|        xlsb|
|2024-07-13_0800|202211_ADMIN.xlsb|        xlsb|
|2024-07-13_0800|202211_ADMIN.xlsb|        xlsb|
|2024-07-13_0800|202211_ADMIN.xlsb|        xlsb|
+---------------+-----------------+------------+
only showing top 5 rows



5. Sumário

In [5]:
validation_results = {
    "csv": csv_output_path,
    "xlsx": xlsx_output_path,
    "xlsb": xlsb_output_path,
}

validation_results

{'csv': 's3a://contracts/bronze/source_type=csv/dataset=controle_medicoes_pagamentos/snapshot_date=2024-07-13_0800/',
 'xlsx': 's3a://contracts/bronze/source_type=xlsx/dataset=controle_medicoes_andamento/snapshot_date=2024-07-13_0800/',
 'xlsb': 's3a://contracts/bronze/source_type=xlsb/dataset=nact/snapshot_date=2024-07-13_0800/'}

5. Isso libera o worker para o próximo notebook.

In [6]:
#spark.stop()

### Batch Validation

In [7]:
# 1. Run Bronze Batch
from src.bronze.batch import run_bronze_batch

bronze_batch_df = run_bronze_batch()

bronze_batch_df["status"].value_counts()

status
SKIPPED    36
Name: count, dtype: int64

In [8]:
# 2. Execution Summary
bronze_batch_df[
    [
        "execution_id",
        "source_file",
        "dataset_name",
        "source_type",
        "snapshot_date",
        "status",
        "duration_seconds",
    ]
].head(10)

,execution_id,source_file,dataset_name,source_type,snapshot_date,status,duration_seconds
0,bronze_20260624_190956_908279e9,ControleMedicoesPagamentos.csv,controle_medicoes_pagamentos,csv,2024-07-13_0800,SKIPPED,0.001622
1,bronze_20260624_190956_908279e9,Exportação_bm_acompanhamento.xlsx,controle_medicoes_andamento,xlsx,2024-07-13_0800,SKIPPED,0.001842
2,bronze_20260624_190956_908279e9,202211_ADMIN.xlsb,nact,xlsb,2024-07-13_0800,SKIPPED,0.001507
3,bronze_20260624_190956_908279e9,Pendências-010223.xlsx,pendências_010223,xlsx,2024-07-13_0800,SKIPPED,0.001195
4,bronze_20260624_190956_908279e9,QEC_5900055119_7_55_77.csv,qec,csv,2024-07-13_0800,SKIPPED,0.000857
5,bronze_20260624_190956_908279e9,QEC_5900074722_7_55_77.csv,qec,csv,2024-07-13_0800,SKIPPED,0.000968
6,bronze_20260624_190956_908279e9,QEC_5900083950_7_55_77.csv,qec,csv,2024-07-13_0800,SKIPPED,0.000889
7,bronze_20260624_190956_908279e9,QEC_5900086165_7_55_77.csv,qec,csv,2024-07-13_0800,SKIPPED,0.000810
8,bronze_20260624_190956_908279e9,QEC_5900086169_7_55_77.csv,qec,csv,2024-07-13_0800,SKIPPED,0.000847
9,bronze_20260624_190956_908279e9,AnaliticoProjeto.csv,analitico_projeto,csv,2024-07-13_0800,SKIPPED,0.000888


In [9]:
# 3. Failed Records
bronze_batch_df[
    bronze_batch_df["status"] == "FAILED"
][
    [
        "source_file",
        "snapshot_date",
        "dataset_name",
        "error_message",
    ]
]

,source_file,snapshot_date,dataset_name,error_message


In [10]:
# Validar o log persistido:
from src.config.settings import Settings
import pandas as pd

pd.read_csv(Settings.BRONZE_EXECUTION_LOG_PATH).head(10)

,execution_id,source_file,dataset_name,status,start_time,end_time,duration_seconds,error_message,snapshot_date,source_type,file_hash,bronze_path
0,bronze_20260624_145226_7600ba06,ControleMedicoesPagamentos.csv,controle_medicoes_pagamentos,SUCCESS,2026-06-24T14:52:26.464756+00:00,2026-06-24T14:52:27.413885+00:00,0.949129,NaN,2024-07-13_0800,csv,b73d09f9dc6766eb918f3e1149649f758e0338d0f42f86...,s3a://contracts/bronze/source_type=csv/dataset...
1,bronze_20260624_145245_cf1a7458,ControleMedicoesPagamentos.csv,controle_medicoes_pagamentos,SKIPPED,2026-06-24T14:52:50.189023+00:00,2026-06-24T14:52:50.191407+00:00,0.002384,NaN,2024-07-13_0800,csv,b73d09f9dc6766eb918f3e1149649f758e0338d0f42f86...,NaN
2,bronze_20260624_145245_cf1a7458,Exportação_bm_acompanhamento.xlsx,controle_medicoes_andamento,SUCCESS,2026-06-24T14:52:50.191624+00:00,2026-06-24T14:53:01.947724+00:00,11.756100,NaN,2024-07-13_0800,xlsx,84df6ac3cb58f7ca789bf83e192ad99369bb1d35c6b41f...,s3a://contracts/bronze/source_type=xlsx/datase...
3,bronze_20260624_145245_cf1a7458,202211_ADMIN.xlsb,nact,SUCCESS,2026-06-24T14:53:01.947961+00:00,2026-06-24T14:53:14.508814+00:00,12.560853,NaN,2024-07-13_0800,xlsb,5d05acde002b287d9068b5b4b8dc34dc04609d1cc16cbb...,s3a://contracts/bronze/source_type=xlsb/datase...
4,bronze_20260624_145245_cf1a7458,Pendências-010223.xlsx,pendências_010223,SUCCESS,2026-06-24T14:53:14.509125+00:00,2026-06-24T14:53:15.560370+00:00,1.051245,NaN,2024-07-13_0800,xlsx,3cca4c48eaa038cee8bd806f6cae89ca1de1e6ee0e9996...,s3a://contracts/bronze/source_type=xlsx/datase...
5,bronze_20260624_145245_cf1a7458,QEC_5900055119_7_55_77.csv,qec,SUCCESS,2026-06-24T14:53:15.560937+00:00,2026-06-24T14:53:16.971908+00:00,1.410971,NaN,2024-07-13_0800,csv,f8a0265c96ffd7085b4e65eff75c24ba5d3439daeabbc2...,s3a://contracts/bronze/source_type=csv/dataset...
6,bronze_20260624_145245_cf1a7458,QEC_5900074722_7_55_77.csv,qec,SUCCESS,2026-06-24T14:53:16.972197+00:00,2026-06-24T14:53:18.192436+00:00,1.220239,NaN,2024-07-13_0800,csv,705844e05903b043b8ffcbe1f4889db5e08aea5370d378...,s3a://contracts/bronze/source_type=csv/dataset...
7,bronze_20260624_145245_cf1a7458,QEC_5900083950_7_55_77.csv,qec,SUCCESS,2026-06-24T14:53:18.192738+00:00,2026-06-24T14:53:19.136027+00:00,0.943289,NaN,2024-07-13_0800,csv,e9d67a12cd2f3754ab7e647f0e330ff373509e7fe74643...,s3a://contracts/bronze/source_type=csv/dataset...
8,bronze_20260624_145245_cf1a7458,QEC_5900086165_7_55_77.csv,qec,SUCCESS,2026-06-24T14:53:19.136501+00:00,2026-06-24T14:53:20.111475+00:00,0.974974,NaN,2024-07-13_0800,csv,acf1d09ec8d814ce474f28a371b3b2d885d5c183130b9c...,s3a://contracts/bronze/source_type=csv/dataset...
9,bronze_20260624_145245_cf1a7458,QEC_5900086169_7_55_77.csv,qec,SUCCESS,2026-06-24T14:53:20.111709+00:00,2026-06-24T14:53:21.011548+00:00,0.899839,NaN,2024-07-13_0800,csv,5978cdd5de1fa7b267ff214b6e0e4f56b3b310810fdf9b...,s3a://contracts/bronze/source_type=csv/dataset...


### Idempotência
Ex: arquivo processado 10 vezes

Em produção isso é inaceitável.
se source_file + snapshot_date + file_hash já foi processado com SUCCESS
→ não reprocessar


In [11]:
from src.bronze.batch import run_bronze_batch

bronze_batch_df = run_bronze_batch(force_reprocess=False)

bronze_batch_df["status"].value_counts()

status
SKIPPED    36
Name: count, dtype: int64

### Quality Log / Reject Tracking

Objetivo: registrar problemas técnicos durante leitura
sem quebrar o batch inteiro

In [12]:
# Validação
bronze_batch_df = run_bronze_batch(force_reprocess=True)

bronze_batch_df["status"].value_counts()

26/06/24 19:10:09 WARN TaskSetManager: Stage 11 contains a task of very large size (1043 KiB). The maximum recommended task size is 1000 KiB.
Skipping line 479: Expected 13 fields in line 479, saw 15                       
26/06/24 19:10:32 WARN TaskSetManager: Stage 23 contains a task of very large size (1047 KiB). The maximum recommended task size is 1000 KiB.
Skipping line 479: Expected 13 fields in line 479, saw 15                       
26/06/24 19:10:54 WARN TaskSetManager: Stage 35 contains a task of very large size (1047 KiB). The maximum recommended task size is 1000 KiB.
Skipping line 479: Expected 13 fields in line 479, saw 15


status
SUCCESS    36
Name: count, dtype: int64

In [13]:
from src.config.settings import Settings
import pandas as pd

quality_log_df = pd.read_csv(Settings.BRONZE_QUALITY_LOG_PATH)

quality_log_df.tail(10)

,logged_at,source_file,snapshot_date,source_path,issue_type,issue_description,line_number,severity
0,2026-06-24T19:07:22.710310+00:00,Relatorio_geral_irregularidades.csv,2024-07-13_0800,/app/data/raw_local/RAW/2024-07-13_0800/RELATO...,BAD_CSV_LINE_SKIPPED,CSV contained at least one malformed line skip...,479,WARNING
1,2026-06-24T19:07:43.775098+00:00,Relatorio_geral_irregularidades.csv,2026-06-06_0800,/app/data/raw_local/RAW/2026-06-06_0800/RELATO...,BAD_CSV_LINE_SKIPPED,CSV contained at least one malformed line skip...,479,WARNING
2,2026-06-24T19:08:03.621937+00:00,Relatorio_geral_irregularidades.csv,2026-06-13_0800,/app/data/raw_local/RAW/2026-06-13_0800/RELATO...,BAD_CSV_LINE_SKIPPED,CSV contained at least one malformed line skip...,479,WARNING
3,2026-06-24T19:10:19.047408+00:00,Relatorio_geral_irregularidades.csv,2024-07-13_0800,/app/data/raw_local/RAW/2024-07-13_0800/RELATO...,BAD_CSV_LINE_SKIPPED,CSV contained at least one malformed line skip...,479,WARNING
4,2026-06-24T19:10:41.352972+00:00,Relatorio_geral_irregularidades.csv,2026-06-06_0800,/app/data/raw_local/RAW/2026-06-06_0800/RELATO...,BAD_CSV_LINE_SKIPPED,CSV contained at least one malformed line skip...,479,WARNING
5,2026-06-24T19:11:01.447216+00:00,Relatorio_geral_irregularidades.csv,2026-06-13_0800,/app/data/raw_local/RAW/2026-06-13_0800/RELATO...,BAD_CSV_LINE_SKIPPED,CSV contained at least one malformed line skip...,479,WARNING


### Retry Mechanism

BronzePipeline.run()
↓
try
↓
falhou?
↓
retry 3 vezes
↓
continua falhando?
↓
FAILED definitivo

Integrar Retry no Batch

pipeline.run()
↓
falhou
↓
retry automático (até 3 vezes)
↓
se continuar falhando → FAILED

In [4]:
import importlib
import src.bronze.execution_logger
import src.bronze.batch

importlib.reload(src.bronze.execution_logger)
importlib.reload(src.bronze.batch)

from src.bronze.batch import run_bronze_batch

In [8]:
bronze_batch_df = run_bronze_batch(force_reprocess=True)

bronze_batch_df["status"].value_counts()

bronze_batch_df[["source_file", "status", "retry_count"]].head()

26/06/24 19:42:42 WARN TaskSetManager: Stage 74 contains a task of very large size (1043 KiB). The maximum recommended task size is 1000 KiB.
Skipping line 479: Expected 13 fields in line 479, saw 15
26/06/24 19:43:01 WARN TaskSetManager: Stage 86 contains a task of very large size (1047 KiB). The maximum recommended task size is 1000 KiB.
Skipping line 479: Expected 13 fields in line 479, saw 15                       
26/06/24 19:43:19 WARN TaskSetManager: Stage 98 contains a task of very large size (1047 KiB). The maximum recommended task size is 1000 KiB.
Skipping line 479: Expected 13 fields in line 479, saw 15


,source_file,status,retry_count
0,ControleMedicoesPagamentos.csv,SUCCESS,0
1,Exportação_bm_acompanhamento.xlsx,SUCCESS,0
2,202211_ADMIN.xlsb,SUCCESS,0
3,Pendências-010223.xlsx,SUCCESS,0
4,QEC_5900055119_7_55_77.csv,SUCCESS,0


### Processed Files Manifest